# 1.1 goad_toolkit 101: pipelines, steps, and typehints

The goal of this notebook is to help you become comfortable with `Pipeline` and
`TransformBase` before `01.2-irc-chat.ipynb` puts them to work on something real, so this
notebook is short, and has no findings to defend.

If subclassing is still new — what `class Child(Parent):` actually buys you — read
[00.1-classes](../lesson0/00.1-classes.ipynb) first; this notebook picks up right where that
one leaves off, with `TransformBase` and `Pipeline` specifically.

In [ ]:
import pandas as pd
from goad_toolkit.datatransforms import Pipeline, TransformBase
from notebooktester import param

## A pipeline is a sequence of steps you can print

Say you are cleaning up an orders table: apply tax, flag the bulk orders, maybe more later.
Written as loose cells, nothing records that these steps belong together, or in what order
they ran. A `Pipeline` is exactly that record — a list of named steps you build once and can
hand to any dataframe shaped the way the steps expect.

In [ ]:
orders = pd.DataFrame({
    "item": ["mug", "lamp", "mug", "desk", "mug"],
    "qty": [3, 1, 24, 2, 500],
    "price": [8.50, 42.00, 8.50, 175.00, 8.50],
})
orders

## The contract: one method, `transform`

A step is any class that inherits from `TransformBase` and implements one method:

```python
class MyStep(TransformBase):
    def transform(self, data: pd.DataFrame, column: str) -> pd.DataFrame:
        ...
        return data
```

all transformations take in a pandas dataframe and a column to impact, and output a transformed or updated dataframe.

Whatever keyword arguments you pass when you register a step — lets say you have `column="price"` —
`TransformBase.__init__` stores them, and this allows you to let `__call__` unpack them into `transform`. `TransformBase` 
will always check that named `column` variables actually exists in the frame. You write the middle; the parent writes
the plumbing around it.

In [ ]:
class AddTax(TransformBase):
    """Add VAT to a price column."""

    def transform(self, data: pd.DataFrame, column: str, rate: float) -> pd.DataFrame:
        data[f"{column}_with_tax"] = data[column] * (1 + rate)
        return data


class FlagBulk(TransformBase):
    """Flag rows whose quantity crosses a threshold."""

    def transform(self, data: pd.DataFrame, column: str, threshold: int) -> pd.DataFrame:
        data["is_bulk"] = data[column] >= threshold
        return data

A step also works on its own, no `Pipeline` in sight — this is the same dunder-method call
from 00.1-classes' `WordCount`:

In [ ]:
AddTax(column="price", rate=0.21)(orders) # TransformBase will check if "price" actually exists

## Composing steps into a `Pipeline`

`Pipeline.add` takes the *class*, not an instance — the step is only built and run inside
`.apply()`. That is what makes the pipeline a description you can inspect and reuse, rather
than a list of already-built objects sitting in memory.

In [ ]:
pipeline = Pipeline()
pipeline.add(AddTax, column="price", rate=0.21)
pipeline.add(FlagBulk, column="qty", threshold=10)

result = pipeline.apply(orders)
result

In [ ]:
print(pipeline)

`print(pipeline)` is the whole point: it answers "what did you do to this data" without you
having to reread the notebook, and `pipeline.apply` answers it identically on any frame shaped
like `orders` — next month's export, a filtered subset, whatever you hand it. That is what
makes writing the step worth it the first time: you are not describing this one run, you are
describing the transformation itself.

## Reading the contract in the type hints

Look again at `AddTax.transform`'s signature:

```python
def transform(self, data: pd.DataFrame, column: str, rate: float) -> pd.DataFrame:
```

Every hint there is a promise, read left to right: hand this a `DataFrame`, a column name and
a rate, and it hands a `DataFrame` back. You can tell that from the signature alone, without
reading the body — which is the entire value of a type hint. It is documentation that sits
exactly where you need it, rather than in a docstring you have to scroll to, or a comment
nobody kept up to date.

## Building your own step, on real data

Everything above happens on rows of an imaginary orders table. `01.2-irc-chat.ipynb` derives a
step of exactly the same shape — `RegexFeature` — for a real problem: pulling a timestamp, an
author or a URL out of a wall of text. Same `TransformBase`, same one method, same reason to
write it as a class instead of a loose cell.

> **Your turn.** Add one more step to the pipeline above: a class that sets a boolean column
> when `qty` crosses some threshold you choose — call it `FreeShipping`, or anything else.
> Register it with `pipeline.add(...)` in the cell below and re-run; the assert checks that a
> new column showed up.

In [ ]:
# >>> Your turn: define a step and register it, then re-run this cell. <<<
#
# class FreeShipping(TransformBase):
#     def transform(self, data: pd.DataFrame, column: str, threshold: int) -> pd.DataFrame:
#
#         # your implemetation here
#
#         return data
#
# pipeline.add(FreeShipping, column="qty", threshold=50)

your_turn = pipeline.apply(orders)
new_columns = set(your_turn.columns) - set(result.columns)

MIN_NEW_STEPS = param(1, test=0)
assert len(new_columns) >= MIN_NEW_STEPS, (  # noqa: S101 -- the check *is* the point here
    f"add at least {MIN_NEW_STEPS} more pipeline.add(...) step above, so pipeline.apply "
    f"produces a column beyond {sorted(result.columns)}"
)
print(f"new column(s): {sorted(new_columns) or 'none yet -- add a step above'}")